# CBFV generation demo

Turning chemical formulae into features with the CBFV package.

You need a dataframe with a formula column and a target column. The notebook generates the features and joins them back on.

## Imports

pandas for dataframes, CBFV for the featurisation, pathlib for file paths.

In [ ]:
import pandas as pd 
from CBFV import composition  # package that turns formulae into features
from pathlib import Path

## Load the data

Read in the cleaned thermoelectric dataset: measurement temperature, formula, log10 conductivity, class label, source, and entry. Only the formula is needed for featurisation; the rest are kept for reference.

In [ ]:
data_path = Path.cwd().parent / "data" / "te_cleaned.xlsx"  # replace this with the path to your data file
data = pd.read_excel(data_path)

# We are only interested in modelling pure Semiconductors/Metals 
data = data[(data["class"] == "Semiconductor") | (data["class"] == "Metal")]
# Ensure these are converted to 0s and 1s for the model
data["class"] = data["class"].map({"Semiconductor": 0, "Metal": 1})
data = data.reset_index(drop=True)
# We have to rename the columns according to the model framework
data.columns = ["temp", "formula", "target", "class", "source", "entry"]
data.head()

In [ ]:
cbfv = "magpie" # default cbfv, alternative options: "oliynyk", "jarvis", "mat2vec", "onehot", "custom"
export_folder = data_path = Path.cwd().parent / "data"  # replace this with the path to your export folder
input = data[["formula", "target"]]
X, _, formulae, skipped = composition.generate_features(input,
                                                        elem_prop=cbfv,
                                                        drop_duplicates=False,
                                                        extend_features=True,
                                                        sum_feat=True,
                                                        )
data = pd.concat([data, X], axis=1)

In [ ]:
# Export to CSV for later use
data.to_csv(export_folder / "magpie_data.csv", index=False)